# Waymo Open Dataset v2 — Exploratory Data Analysis

This notebook streams data **directly from GCS** (no local download required).  
It covers:

1. [Dataset Statistics](#1.-Dataset-Statistics) — frame & object-class counts  
2. [Camera Frames + 2-D Boxes](#2.-Camera-Frames-with-2-D-Bounding-Boxes)  
3. [LiDAR — Bird's-Eye View](#3.-LiDAR-Point-Cloud-—-Bird's-Eye-View)  
4. [LiDAR — 3-D Scene (Open3D)](#4.-LiDAR-Point-Cloud-—-3-D-Scene)  
5. [LiDAR → Camera Projection](#5.-LiDAR-to-Camera-Projection)

**Prerequisite:** authenticate with GCS before running:
```bash
gcloud auth application-default login
```

## 0. Setup

In [ ]:
import sys, os
# Make the modules/ package importable from the notebooks/ directory
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import open3d as o3d
from IPython.display import display
from waymo_open_dataset import v2

from modules.WaymoOpenDataset import ToolKit, CAMERA_NAMES, LABEL_TYPES
from modules import visualize as viz

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

## Segment selection

Run the cell below to list available segments, then paste one into `SEGMENT`.

In [ ]:
SPLIT = 'training'   # 'training' | 'validation' | 'testing'

toolkit = ToolKit(split=SPLIT)
segments = toolkit.list_segments()
print(f'Found {len(segments)} segments in the {SPLIT} split.  First 5:')
for s in segments[:5]:
    print(' ', s)

In [ ]:
# ── Pick a segment ────────────────────────────────────────────────────────
SEGMENT = segments[0]   # change to any context name from the list above
# ──────────────────────────────────────────────────────────────────────────

toolkit.assign_segment(SEGMENT)
timestamps = toolkit.get_timestamps()
print(f'Segment : {SEGMENT}')
print(f'Frames  : {len(timestamps)}')
print(f'Duration: {(timestamps[-1] - timestamps[0]) / 1e6:.1f} s')

---
## 1. Dataset Statistics

In [ ]:
# Load the full lidar_box component for this segment (all frames)
all_boxes = toolkit.load_all_boxes_df()

# Map int type → string label using the v2 component API
def _type_label(row):
    box = v2.LiDARBoxComponent.from_dict(row)
    return LABEL_TYPES.get(box.type, 'TYPE_UNKNOWN')

all_boxes['label'] = all_boxes.apply(_type_label, axis=1)

print(f'Total 3-D box annotations across {len(timestamps)} frames: {len(all_boxes):,}')
print()
print(all_boxes['label'].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── (A) Class distribution pie ───────────────────────────────────────────
counts  = all_boxes['label'].value_counts()
colors  = [viz.LABEL_COLORS_RGB[k] for k in counts.index]
axes[0].pie(counts, labels=[k.replace('TYPE_', '') for k in counts.index],
            colors=colors, autopct='%1.1f%%', startangle=140)
axes[0].set_title('Object class distribution (all frames)')

# ── (B) Objects per frame histogram ──────────────────────────────────────
per_frame = all_boxes.groupby('key.frame_timestamp_micros').size()
axes[1].hist(per_frame, bins=20, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Number of 3-D boxes in frame')
axes[1].set_ylabel('Frame count')
axes[1].set_title('3-D boxes per frame')

fig.suptitle(f'Segment: {SEGMENT[:30]}…', fontsize=9, color='gray')
plt.tight_layout()
plt.show()

---
## 2. Camera Frames with 2-D Bounding Boxes

In [ ]:
# Pick a frame timestamp (use the slider index below)
FRAME_IDX   = 0     # change to explore different frames
CAMERA_ID   = 1     # 1=FRONT, 2=FRONT_LEFT, 3=FRONT_RIGHT, 4=SIDE_LEFT, 5=SIDE_RIGHT

ts  = timestamps[FRAME_IDX]
img = toolkit.load_camera_frame(ts, CAMERA_ID)
boxes_df = toolkit.load_camera_boxes(ts, CAMERA_ID)

annotated = viz.draw_camera_boxes(img, boxes_df)

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
ax.axis('off')
ax.set_title(f'{CAMERA_NAMES[CAMERA_ID]} — frame {FRAME_IDX}  |  '
             f'{len(boxes_df)} boxes')
plt.tight_layout()
plt.show()

In [ ]:
# All 5 cameras in one grid for the same frame
cam_ids  = [1, 2, 3, 4, 5]
cam_labels = [CAMERA_NAMES[c] for c in cam_ids]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, cam_id, cam_label in zip(axes, cam_ids, cam_labels):
    try:
        frame = toolkit.load_camera_frame(ts, cam_id)
        boxes = toolkit.load_camera_boxes(ts, cam_id)
        ann   = viz.draw_camera_boxes(frame, boxes)
        ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{cam_label}  ({len(boxes)} boxes)', fontsize=9)
    except IndexError:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
    ax.axis('off')

axes[-1].axis('off')   # hide unused 6th subplot
fig.suptitle(f'Frame {FRAME_IDX} — all cameras', fontsize=11)
plt.tight_layout()
plt.show()

---
## 3. LiDAR Point Cloud — Bird's-Eye View

In [ ]:
points_list = toolkit.load_lidar_points(ts)
lidar_boxes = toolkit.load_lidar_boxes(ts)

total_pts = sum(len(p) for p in points_list)
print(f'Lasers loaded : {len(points_list)}')
print(f'Total points  : {total_pts:,}')
print(f'3-D boxes     : {len(lidar_boxes)}')

In [ ]:
fig = viz.plot_bev(points_list, boxes_df=lidar_boxes, range_m=75.0)
plt.show()

---
## 4. LiDAR Point Cloud — 3-D Scene (Open3D)

> **Note:** `o3d.visualization.draw_geometries()` opens a **separate interactive window**.  
> On a headless machine (remote SSH / Colab), use the BEV plot above instead.

In [ ]:
geometries = viz.build_open3d_scene(points_list, boxes_df=lidar_boxes)
print(f'Scene objects: {len(geometries)} (PointCloud + {len(geometries)-2} box LineSet + origin frame)')

# Opens an interactive 3-D viewer (separate window)
o3d.visualization.draw_geometries(
    geometries,
    window_name='Waymo LiDAR',
    width=1280, height=720,
    point_show_normal=False,
)

---
## 5. LiDAR to Camera Projection

Project every LiDAR point into the **FRONT camera** image plane.  
Points are coloured by depth using the Turbo colour map (blue = near, red = far).

In [ ]:
CAM_ID_PROJ = 1   # 1 = FRONT

cam_img    = toolkit.load_camera_frame(ts, CAM_ID_PROJ)
cam_calib  = toolkit.load_camera_calibration(CAM_ID_PROJ)

overlay = viz.draw_lidar_on_camera(
    cam_img, points_list, cam_calib, max_depth=75.0
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].imshow(cv2.cvtColor(cam_img, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'{CAMERA_NAMES[CAM_ID_PROJ]} — camera only')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'{CAMERA_NAMES[CAM_ID_PROJ]} — LiDAR depth overlay')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Combined: camera boxes + LiDAR depth overlay
cam_boxes_front = toolkit.load_camera_boxes(ts, CAM_ID_PROJ)
combined = viz.draw_camera_boxes(overlay, cam_boxes_front)

fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(cv2.cvtColor(combined, cv2.COLOR_BGR2RGB))
ax.set_title(f'{CAMERA_NAMES[CAM_ID_PROJ]} — 2-D boxes + LiDAR depth  |  frame {FRAME_IDX}')
ax.axis('off')
plt.tight_layout()
plt.show()